# Chapter 19 — pandas DataFrames and NumPy for an interactive analytics application

**Book role:** Analytics foundations  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Understand Series, DataFrame, indexes, dtypes, and vectorization.
- Create descriptive summaries and grouped metrics.
- Design filter and aggregation services for a Tkinter GUI.
- Avoid row-wise loops when vectorized operations are clearer.

## 1. A DataFrame is a labeled analytical table

A pandas DataFrame is a two-dimensional labeled data structure with row and column axes. Columns can have different dtypes; the index labels rows and need not be a unique integer range. Use `.head()`, `.info()`, `.describe()`, `.shape`, `.dtypes`, and `.columns` to inspect a source before building a user interface around it.

Avoid assuming every imported column is correctly typed. Dates can arrive as strings, amounts may be stored as text because of currency symbols, and identifiers with leading zeros may be inferred as integers. Use deliberate import options and normalize types in a cleaning stage. Keep identifiers distinct from measures: IDs should not be summed or averaged simply because they are numeric.

### Worked example 1: Create measures and grouped summaries

The calculation is expressed at the transaction-line level, then grouped by region. Confirm whether one row represents an order or an order line before choosing the denominator for any order-level metric.

In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "region": ["North", "South", "North", "West"],
    "quantity": [2, 1, 3, 2],
    "unit_price": [100.0, 250.0, 100.0, 120.0],
    "unit_cost": [60.0, 150.0, 60.0, 75.0],
})
sales = sales.assign(
    revenue=sales["quantity"] * sales["unit_price"],
    cost=sales["quantity"] * sales["unit_cost"],
)
sales["profit"] = sales["revenue"] - sales["cost"]
summary = (sales.groupby("region", as_index=False)
           .agg(orders=("order_id", "nunique"), revenue=("revenue", "sum"), profit=("profit", "sum"))
           .sort_values("revenue", ascending=False))
print(summary)

## 2. Vectorization, masks, and grouped analysis

Vectorized pandas operations compute on a Series or DataFrame without an explicit Python loop per row. Boolean masks filter records; `.assign()` creates derived columns; `.groupby()` produces grouped summaries; `.agg()` lets you name multiple measures; and `.sort_values()` prepares results for a table or chart. NumPy adds efficient numerical arrays, broadcasting, and mathematical functions for numerical analysis.

A Tkinter filter panel should translate user choices into masks and grouping parameters. For example, a category selection yields a boolean mask, then a service calculates revenue and profit. Do not perform heavy computation within every widget trace, and do not use `iterrows()` as the default calculation technique. Loops still have valid uses, but vectorized operations are often faster and simpler for columnar work.

### Worked example 2: Build a testable filter and KPI service

The returned object includes a DataFrame for table rendering and plain summary values for KPI labels. The service rejects an incompatible schema instead of failing later inside a widget update.

In [ ]:
from dataclasses import dataclass
import pandas as pd

@dataclass(frozen=True)
class Filters:
    region: str = "All"
    minimum_revenue: float = 0.0


def summarize(frame: pd.DataFrame, filters: Filters) -> dict:
    required = {"region", "revenue"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    result = frame.copy()
    if filters.region != "All":
        result = result[result["region"].eq(filters.region)]
    result = result[result["revenue"].ge(filters.minimum_revenue)]
    return {
        "rows": int(len(result)),
        "revenue": float(result["revenue"].sum()),
        "average_revenue": float(result["revenue"].mean()) if not result.empty else None,
        "records": result,
    }

print(summarize(sales.assign(revenue=sales["quantity"] * sales["unit_price"]), Filters("North", 100)))

## 3. Statistics need a business definition

`sum`, `mean`, `median`, `min`, `max`, `count`, and `nunique` answer different questions. Does average order value mean total sales divided by number of orders, or mean revenue per line? Does a cancelled transaction contribute zero sales or disappear from the denominator? Should profit equal revenue minus cost, and are returns negative quantities? These are domain decisions, not merely syntax choices.

Name aggregation outputs clearly and test them on a tiny dataset whose expected values can be calculated by hand. For dashboards, expose row counts and filter scope. A KPI without the date range, currency, and included statuses can be mathematically correct but misleading.

### Worked example 3: Use NumPy for numerical operations

NumPy makes the missing-value policy explicit. Standardization is not automatically desirable for every analytics problem; preserve the original measure and explain transformed units.

In [ ]:
import numpy as np

values = np.array([12.0, 15.0, np.nan, 18.0])
valid = values[~np.isnan(values)]
print("Mean of valid observations:", float(np.mean(valid)))
print("Population standard deviation:", float(np.std(valid)))

# Example: normalize a numeric feature only if the analytical question calls for it.
if valid.size and np.std(valid) > 0:
    standardized = (valid - np.mean(valid)) / np.std(valid)
    print("Standardized values:", standardized)

## Guided lab: Lab: KPI service for a dashboard

Use the sample sales file to implement KPIs for net revenue, gross profit, profit margin, transaction count, unique customers, and average order value. Define precisely how cancelled rows, missing costs, returns, and duplicate order IDs are handled. Build one Tkinter view that renders the service result, but write unit tests against the service first.

## Review questions and practice

1. Why can a numeric identifier be a poor candidate for an average?
2. What is the difference between row count and unique order count?
3. Why should aggregate names be explicit?
4. What does vectorization generally improve?
5. What should a KPI specify besides its numeric value?

## Further reading

- pandas user guide: https://pandas.pydata.org/docs/user_guide/
- NumPy quickstart: https://numpy.org/doc/stable/user/quickstart.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.